In [2]:
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import cm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchsummary

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Training on device: {device}")

Training on device: cuda


In [5]:
# We start with loading the .ssm data in this cell to keep the data loaded in the following
def load_full_dataset(dataset='Train'):
    # 1) Path Definition
    # base directory for the chosen dataset
    base_path = Path('../data/Challenge-ABC') / dataset
    if not base_path.exists(): # verification
        print(f"Error: Dataset folder {base_path} not found, check 'dataset' argument.")
        return None
    ssm_dir = base_path / 'SSM_Challenge-ABC'
    lb_dir = base_path / 'lb'

    X_list = []
    y_list = []

    ssm_files = list(ssm_dir.glob('*.ssm'))
    total_files = len(ssm_files)

    for i, ssm_file in enumerate(ssm_files, 1):
        file_id = ssm_file.stem
        lb_file = lb_dir / f"{file_id}.lb"

        # check that the labels file exists
        if not lb_file.exists():
            print(f"Error: Didn't find labels file for {file_id}.ply -> ignored.")
            continue

        features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 20, 16)
        labels = np.loadtxt(lb_file, dtype='int')

        if len(labels) == len(features):
            X_list.append(features)
            y_list.append(labels)
        else:
            print(f"Error: Dimension error for {file_id} -> ignored.")

        if i % 20 == 0:
            print(f"Loading : {i}/{total_files} files...")

    X = np.vstack(X_list)
    y = np.concatenate(y_list)    
    
    return X, y

print("Train dataset loading...")
X_train, y_train = load_full_dataset('Train')
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

print("\nValidation dataset loading...")
X_val, y_val = load_full_dataset('Validation')
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")


Train dataset loading...
Loading : 20/198 files...
Loading : 40/198 files...
Loading : 60/198 files...
Loading : 80/198 files...
Loading : 100/198 files...
Loading : 120/198 files...
Loading : 140/198 files...
Loading : 160/198 files...
Loading : 180/198 files...
X_train shape: (3174768, 20, 16)
y_train shape: (3174768,)

Validation dataset loading...
Loading : 20/50 files...
Loading : 40/50 files...
X_val shape: (690211, 20, 16)
y_val shape: (690211,)


In [6]:
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

print(type(X_train))
print(type(y_train))
print(type(X_val))
print(type(y_val))

# Transform numpy's arrays to torch tensors:
X_train = torch.tensor(X_train).to(torch.float)
y_train = torch.tensor(y_train)
X_val = torch.tensor(X_val).to(torch.float)
y_val = torch.tensor(y_val)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)


X_train shape: (3174768, 20, 16)
y_train shape: (3174768,)
X_val shape: (690211, 20, 16)
y_val shape: (690211,)
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
torch.Size([3174768, 20, 16]) torch.Size([3174768])
torch.Size([690211, 20, 16]) torch.Size([690211])


In [ ]:
class MLP(nn.Module):
    def __init__(self, num_hidden_1=64, num_hidden_2=64):
        super(MLP, self).__init__()
        self.layer_1 = torch.nn.Linear(320, num_hidden_1)
        self.layer_2 = torch.nn.Linear(num_hidden_1, num_hidden_2)
        self.layer_3 = torch.nn.Linear(num_hidden_2, 2)
        self.num_hidden_1 = num_hidden_1
        self.num_hidden_2 = num_hidden_2

    def forward(self, x):
        # Flatten the input from [batch_size, 20, 16] to [batch_size, 320]
        x = x.view(x.size(0), -1)

        # 1st layer
        out = self.layer_1(x)
        #out = torch.tanh(out) 
        out = torch.sigmoid(out)
        
        # 2nd layer
        out = self.layer_2(out)
        out = torch.sigmoid(out)

        # 3rd/output layer
        out = self.layer_3(out)
        return out

In [8]:
# Check paramters number
model = MLP().to(device)

print(torchsummary.summary(model,input_size=(320,)))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]          20,544
            Linear-2                   [-1, 64]           4,160
            Linear-3                    [-1, 2]             130
Total params: 24,834
Trainable params: 24,834
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.09
Estimated Total Size (MB): 0.10
----------------------------------------------------------------
None
